# 04 — Evaluation

Goal: deep-dive into the best model's performance — confusion matrix, PR curve, threshold tuning, and feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve,
    average_precision_score, roc_auc_score, f1_score
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Model and Test Data

In [ ]:
model = joblib.load('../models/best_model.pkl')

test = pd.read_csv('../data/processed/test.csv')
X_test = test.drop(columns=['Class'])
y_test = test['Class']

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print(f'PR-AUC : {average_precision_score(y_test, y_prob):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

## 2. Confusion Matrix (Default Threshold = 0.5)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'])
plt.title('Confusion Matrix (default threshold=0.5)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))

## 3. Precision-Recall Curve

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
prauc = average_precision_score(y_test, y_prob)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision, label=f'PR-AUC = {prauc:.4f}', color='steelblue')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/pr_curve.png', dpi=150)
plt.show()

## 4. Threshold Tuning

Find the threshold that maximizes F1-score.

In [ ]:
f1_scores = [f1_score(y_test, (y_prob >= t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f'Best threshold: {best_threshold:.4f}')
print(f'Best F1:        {best_f1:.4f}')

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores, color='tomato')
plt.axvline(best_threshold, linestyle='--', color='gray', label=f'Best threshold={best_threshold:.3f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Threshold')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/threshold_tuning.png', dpi=150)
plt.show()

## 5. Confusion Matrix (Best Threshold)

In [ ]:
y_pred_best = (y_prob >= best_threshold).astype(int)
cm_best = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'])
plt.title(f'Confusion Matrix (threshold={best_threshold:.3f})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix_best_threshold.png', dpi=150)
plt.show()

print(classification_report(y_test, y_pred_best, target_names=['Legit', 'Fraud']))

## 6. Feature Importance

In [ ]:
if hasattr(model, 'feature_importances_'):
    importances = pd.Series(model.feature_importances_, index=X_test.columns)
    importances = importances.sort_values(ascending=False).head(20)

    plt.figure(figsize=(8, 6))
    importances.plot(kind='barh', color='steelblue')
    plt.title('Top 20 Feature Importances')
    plt.xlabel('Importance')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('../reports/figures/feature_importance.png', dpi=150)
    plt.show()
else:
    # Logistic Regression: use absolute coefficient values
    coef = pd.Series(np.abs(model.coef_[0]), index=X_test.columns)
    coef = coef.sort_values(ascending=False).head(20)

    plt.figure(figsize=(8, 6))
    coef.plot(kind='barh', color='steelblue')
    plt.title('Top 20 Feature Coefficients (|coef|)')
    plt.xlabel('|Coefficient|')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('../reports/figures/feature_importance.png', dpi=150)
    plt.show()